In [408]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

In [409]:
# 주택 가격 데이터를 기반으로 회귀 모델을 학습하여 주택 가격을 예측 해야함
# 주택 가격 데이터 불러오기
df = pd.read_csv('./dataset/train.csv') #
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [410]:
# LotFrontage 평균값으로 결측치 대체
df['LotFrontage'] = df['LotFrontage'].replace(np.nan, df['LotFrontage'].mean())

In [411]:
# 결측치가 많은 열 삭제 (결측 비율 30% 이상 기준)
mis_data = df.isnull().mean()

cols_drop = mis_data[mis_data >=  0.3].index

df = df.drop(columns=cols_drop)
print("삭제된 열 목록: ",list(cols_drop))


삭제된 열 목록:  ['Alley', 'MasVnrType', 'FireplaceQu', 'PoolQC', 'Fence', 'MiscFeature']


In [412]:
# 불필요한 열(Id) 제거
df = df.drop(columns=['Id'], errors='ignore')

In [413]:
# 범주형 데이터 인코딩(get_dummies 사용)
df = pd.get_dummies(df)

# 결과 확인
print(df.shape)
print(df.head())

(1460, 267)
   MSSubClass  LotFrontage  LotArea  OverallQual  OverallCond  YearBuilt  \
0          60         65.0     8450            7            5       2003   
1          20         80.0     9600            6            8       1976   
2          60         68.0    11250            7            5       2001   
3          70         60.0     9550            7            5       1915   
4          60         84.0    14260            8            5       2000   

   YearRemodAdd  MasVnrArea  BsmtFinSF1  BsmtFinSF2  ...  SaleType_ConLw  \
0          2003       196.0         706           0  ...           False   
1          1976         0.0         978           0  ...           False   
2          2002       162.0         486           0  ...           False   
3          1970         0.0         216           0  ...           False   
4          2000       350.0         655           0  ...           False   

   SaleType_New  SaleType_Oth  SaleType_WD  SaleCondition_Abnorml  \
0    

In [414]:
# 주택가격(SalePrice) 독립변수(X) / 타겟(y) 분리
X = df.drop('SalePrice', axis=1)
y = df['SalePrice']


# 학습/테스트 데이터 분리 (8:2)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)


(1168, 266) (292, 266)
(1168,) (292,)


In [415]:
# 모델 학습 Decision TreeRegressor
model = DecisionTreeRegressor(random_state=42)
model.fit(X_train, y_train)

#예측
y_pred = model.predict(X_test)



In [416]:
# 평가
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE: ", mae)
print("MSE: ", mse)
print("RMSE: ", rmse)
print("R2 Score: ", r2)

MAE:  26486.04794520548
MSE:  1552142683.7123287
RMSE:  39397.24208256625
R2 Score:  0.797643197722417


평가 결과

RMSE가 MSE보다 크게 나타나는 결과로 일부 주택 가격 데이터에서 큰 오차가 존재한다는것을 알 수 있습니다.    
평균 절대 오차(MAE)가 실제 주택 가격과 비교하였을때 약 26484의 오차가 발생합니다.      
결정계수(R2 Score)가 0.7976로 약 80%의 데이터를 올바르게 설명하고 있는 성능이 양호한 모델입니다.      
Decision TreeRegressor을 사용하여 예측시 나쁘지 않은 성능을 보이고 있습니다.   
추후 이상치 제거 또는 과소적합의 위험성이 없게끔 모델에 맞는 규제를 적용하여 오차범위를 더 줄일 수 있을 것으로 예측 됩니다.    